In [5]:
#!/usr/bin/python3
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os 
import asyncio

# Get APIs
load_dotenv()

# llm setup
llm = ChatGroq (
    temperature=0.3,
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile"
)

async def main():
    client = MultiServerMCPClient({
        "math":{
            "command":"python",
            "args": ["J:\2_Open-claw_Agents\langchain_mcp_adapter\server.py"],
            "transport":"stdio"
        },
        "web-search":{
            "url":"http://0.0.0.0:8000/mcp",
            "transport":"streamable_http"
        }
    })

    tools = await client.get_tools()

    def call_model(state: MessagesState):
        response = llm.bind_tools(tools).invoke(state["messages"])
        return {"messages": response}

    # ===Start building the graph====
    builder = StateGraph(MessagesState)

    # add node
    builder.add_node(call_model)
    builder.add_node(ToolNode(tools))

    # connect nodes with edges
    builder.add_edge(START, "call_model")
    builder.add_conditional_edges("call_model", tools_condition)
    builder.add_edge("tools", "call_model")
    builder.add_edge("call_model", END)

    app=builder.compile()
    return app

if __name__ == "__main__":
    app = asyncio.run(main())


RuntimeError: asyncio.run() cannot be called from a running event loop

In [4]:
#Now lets visualize the flow of our graph
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

NameError: name 'app' is not defined